# Cell Painting Parquet Metadata Extraction

This notebook extracts metadata from Cell Painting parquet files generated by cytotable.

## Parameters

These parameters can be overridden by Papermill:

In [ ]:
# Papermill parameters
parquet_file = "../nf-runs/test-profile-outdir/cytotable/2021_04_26_Batch1_BR00117035_A01_1.parquet"
output_json = "metadata.json"

## Setup and Imports

In [ ]:
import pandas as pd
import json
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

## Load Data

In [ ]:
# Load parquet file
print(f"Loading parquet file: {parquet_file}")
df = pd.read_parquet(parquet_file)
print(f"✓ Successfully loaded parquet file")

## File Information

In [ ]:
# Basic file info
file_path = Path(parquet_file)
file_size_mb = file_path.stat().st_size / (1024 * 1024)

print("=" * 80)
print("FILE INFORMATION")
print("=" * 80)
print(f"File name: {file_path.name}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Full path: {file_path.absolute()}")
print()

## DataFrame Overview

In [ ]:
print("=" * 80)
print("DATAFRAME OVERVIEW")
print("=" * 80)
print(f"Number of rows (cells): {df.shape[0]:,}")
print(f"Number of columns (features): {df.shape[1]:,}")
print(f"Total data points: {df.shape[0] * df.shape[1]:,}")
print()

## Column Structure

In [ ]:
# Analyze column structure by prefix
column_prefixes = {}
for col in df.columns:
    if '_' in col:
        prefix = col.split('_')[0]
        if prefix not in column_prefixes:
            column_prefixes[prefix] = []
        column_prefixes[prefix].append(col)

print("=" * 80)
print("COLUMN STRUCTURE BY COMPARTMENT/TYPE")
print("=" * 80)
for prefix in sorted(column_prefixes.keys()):
    count = len(column_prefixes[prefix])
    print(f"{prefix:20s}: {count:5d} columns")
print()

# Show sample columns for each prefix
print("Sample columns for each compartment:")
print("-" * 80)
for prefix in sorted(column_prefixes.keys()):
    sample_cols = column_prefixes[prefix][:3]
    print(f"\n{prefix}:")
    for col in sample_cols:
        print(f"  - {col}")

## Metadata Columns

In [ ]:
# Show all metadata columns
metadata_cols = [col for col in df.columns if col.startswith('Metadata_')]

print("=" * 80)
print(f"METADATA COLUMNS ({len(metadata_cols)} total)")
print("=" * 80)
for col in metadata_cols:
    print(f"  - {col}")
print()

## Data Types

In [ ]:
# Data type summary
dtype_counts = df.dtypes.value_counts()

print("=" * 80)
print("DATA TYPES SUMMARY")
print("=" * 80)
for dtype, count in dtype_counts.items():
    print(f"{str(dtype):20s}: {count:5d} columns")
print()

## Generate Metadata JSON

In [ ]:
# Compile metadata dictionary
metadata = {
    "file_info": {
        "filename": file_path.name,
    },
    "dataframe_overview": {
        "n_rows": int(df.shape[0]),
        "n_columns": int(df.shape[1]),
        "total_data_points": int(df.shape[0] * df.shape[1])
    },
    "column_structure": {
        prefix: {
            "count": len(cols),
            "sample_columns": cols[:3]
        }
        for prefix, cols in sorted(column_prefixes.items())
    },
    "metadata_columns": {
        "count": len(metadata_cols),
        "columns": metadata_cols
    },
    "data_types": {
        str(dtype): int(count)
        for dtype, count in dtype_counts.items()
    }
}

# Save to JSON file
with open(output_json, 'w') as f:
    json.dump(metadata, f, indent=2)

print("=" * 80)
print("METADATA JSON SAVED")
print("=" * 80)
print(f"✓ Metadata saved to: {output_json}")
print(f"✓ Total metadata entries: {len(metadata)}")
print()

## Complete Column List

In [ ]:
print("=" * 80)
print(f"COMPLETE COLUMN LIST ({len(df.columns)} columns)")
print("=" * 80)
for i, col in enumerate(df.columns, 1):
    print(f"{i:4d}. {col}")

## Report Complete

This metadata report was generated using Papermill for automated analysis of Cell Painting parquet files.